# Grading, ID decoding, and review flags

This notebook starts where the detector ends. The detector gives us bubble positions and `fill_score` values. The grading layer turns those into:

- Student ID digits,
- Test ID digits,
- selected answer choices,
- correct/incorrect scoring,
- review flags for blank, multiple, or unclear marks.

The key principle is conservative automation: grade clear marks automatically, but surface uncertain marks for a human instead of pretending the system is more certain than it is.

## 0. Imports and setup

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from omr_bubble_detector import (
    detect_omr_bubbles,
    draw_detection_overlay,
    draw_grading_overlay,
    grade_omr_result,
    read_answer_choices,
    decode_digit_grid,
    warp_sheet,
)

SAMPLES = ROOT / "samples"
OPTIONS = ["A", "B", "C", "D", "E"]

def bgr_to_rgb(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def show_image(image, title=None, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(bgr_to_rgb(image))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12)
    return ax

def demo_answer_key(question_count):
    # For demonstration only. Replace with the real exam key before real scoring.
    return {question: OPTIONS[(question - 1) % len(OPTIONS)] for question in range(1, question_count + 1)}

## 1. Run detection on a sample sheet

For a classroom demo, `cam_sample_50.jpeg` is nice because it has two answer blocks and the full ID header is visible. Try `cam_sample3_100.jpeg` later to show the robust warp recovery.

In [ ]:
SAMPLE_NAME = "cam_sample_50.jpeg"
image_path = SAMPLES / SAMPLE_NAME
image_bgr = cv2.imread(str(image_path))
if image_bgr is None:
    raise FileNotFoundError(image_path)

result = detect_omr_bubbles(image_bgr)
warped_bgr, _ = warp_sheet(image_bgr)
detected_overlay = draw_detection_overlay(warped_bgr, result, draw_labels=True)

print(f"Sample: {image_path.relative_to(ROOT)}")
print(f"Detected questions: {result['answers']['question_count']}")
print(f"Rows per answer group: {result['answers'].get('rows_per_group')}")

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
show_image(image_bgr, "Original", axes[0])
show_image(detected_overlay, "Detection overlay", axes[1])
plt.tight_layout()

## 2. Decode Student ID and Test ID

Each digit position is a small 0-9 multiple-choice column. The decoder compares the darkest bubble in that column with the rest of the column. It returns a digit when the mark is clear; otherwise it uses symbols:

- `_` = blank digit,
- `*` = multiple marked digits,
- `?` = unclear mark.

In [ ]:
student_reading = decode_digit_grid(result["student_id"])
test_reading = decode_digit_grid(result["test_id"])

print("Student ID:", student_reading["value"], "complete:", student_reading["complete"])
print("Test ID:   ", test_reading["value"], "complete:", test_reading["complete"])

display(pd.DataFrame(student_reading["positions"]))
display(pd.DataFrame(test_reading["positions"]))

## 3. Read answer choices from `fill_score`

For each question, the detector has five bubbles: A, B, C, D, and E. The grading layer ranks those five `fill_score` values. A selected answer needs to be meaningfully darker than the row baseline and clearly ahead of the second-darkest bubble.

In [ ]:
answer_reading = read_answer_choices(result["answers"])
responses_df = pd.DataFrame(answer_reading["responses"])

display(pd.Series(answer_reading["counts"], name="answer_status_counts"))
display(responses_df.head(12))

## 4. Grade against an answer key

The demo key below is deliberately simple: A, B, C, D, E, then repeat. It is useful for testing the workflow and overlay. For the real exam, replace this dictionary with the true key.

In [ ]:
answer_key = demo_answer_key(result["answers"]["question_count"])
grading = grade_omr_result(result, answer_key)
result["grading"] = grading

pd.Series(grading["summary"])

In [ ]:
review_df = pd.DataFrame(grading["questions"])
review_columns = [
    "question",
    "detected_answer",
    "correct_answer",
    "outcome",
    "status",
    "confidence",
    "top_fill_score",
    "second_fill_score",
]
review_df[review_columns].head(20)

## 5. Graded overlay

The grading overlay is meant to be explainable at a glance:

- blue ring = the correct answer from the key,
- green ring/check = selected and correct,
- red ring/cross = selected and wrong,
- orange ring = review case.

In [ ]:
graded_overlay = draw_grading_overlay(warped_bgr, result, draw_labels=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
show_image(detected_overlay, "Detection overlay", axes[0])
show_image(graded_overlay, "Grading overlay", axes[1])
plt.tight_layout()

## 6. One-question evidence plot

This plot is useful in a presentation because it shows that the grade comes from a simple, inspectable comparison. We are not asking the audience to trust a black box; each row has five scores.

In [ ]:
QUESTION_TO_INSPECT = 1

question_bubbles = [
    bubble for bubble in result["answers"]["bubbles"]
    if bubble["question"] == QUESTION_TO_INSPECT
]
question_bubbles = sorted(question_bubbles, key=lambda item: item["option_index"])
question_row = review_df.loc[review_df["question"] == QUESTION_TO_INSPECT].iloc[0]

options = [bubble["option"] for bubble in question_bubbles]
scores = [bubble["fill_score"] for bubble in question_bubbles]
colors = []
for option in options:
    if option == question_row["correct_answer"] and option == question_row["detected_answer"]:
        colors.append("#22a35a")
    elif option == question_row["detected_answer"]:
        colors.append("#d33f3f")
    elif option == question_row["correct_answer"]:
        colors.append("#2b7bbb")
    else:
        colors.append("#b8b8b8")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(options, scores, color=colors)
ax.set_ylim(0, max(scores) * 1.20)
ax.set_ylabel("fill_score")
ax.set_title(
    f"Question {QUESTION_TO_INSPECT}: detected {question_row['detected_answer']} | "
    f"key {question_row['correct_answer']} | outcome {question_row['outcome']}"
)
for option, score in zip(options, scores):
    ax.text(option, score + 0.01, f"{score:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()

## 7. Dashboard-style summary figure

This gives you one compact visual for the professor: source photo, detected sheet, graded sheet, and the score/review summary.

In [ ]:
summary = grading["summary"]
identity = grading["identity"]

fig = plt.figure(figsize=(15, 10))
grid = fig.add_gridspec(2, 3, height_ratios=[1, 1])

ax1 = fig.add_subplot(grid[0, 0])
show_image(image_bgr, "Original", ax1)

ax2 = fig.add_subplot(grid[0, 1])
show_image(detected_overlay, "Detected bubbles", ax2)

ax3 = fig.add_subplot(grid[0, 2])
show_image(graded_overlay, "Graded overlay", ax3)

ax4 = fig.add_subplot(grid[1, :])
ax4.axis("off")
summary_text = (
    f"Student ID: {identity['student_id']['value']}\n"
    f"Test ID: {identity['test_id']['value']}\n\n"
    f"Score: {summary['correct']} / {summary['keyed_questions']} "
    f"({summary['score_percent']:.2f}% with the demo key)\n"
    f"Incorrect: {summary['incorrect']}    Blank: {summary['blank']}    "
    f"Multiple: {summary['multiple']}    Unclear: {summary['unclear']}\n"
    f"Needs review: {summary['needs_review']}"
)
ax4.text(0.02, 0.82, summary_text, fontsize=18, va="top", family="monospace")
ax4.text(
    0.02,
    0.18,
    "Presentation note: the demo key is only for workflow validation. The same code accepts the real exam key in Streamlit.",
    fontsize=12,
    color="#444444",
)

plt.tight_layout()

## 8. Functions worth mentioning in the presentation

| Function | Role |
|---|---|
| `read_answer_choices` | Converts five bubble scores per question into selected/blank/multiple/unclear. |
| `decode_digit_grid` | Converts each 0-9 digit column into Student ID or Test ID characters. |
| `grade_omr_result` | Compares detected answers with an answer key and builds a review summary. |
| `draw_grading_overlay` | Draws the key and grading result directly on the warped sheet. |
| Streamlit dashboard | Makes the same outputs interactive for demos and manual review. |
